# EC AI Study: 입문용 Baseline

고객의 과거 금융 이용 정보로 다음 기간의 연체 여부를 예측합니다. 기본 전처리와 로지스틱 회귀 모델 하나로 학습부터 제출까지 진행합니다. 위에서부터 순서대로 실행하세요.

## 1. 패키지 준비

README의 설치 명령을 먼저 실행하세요. `pandas`는 표 형태의 데이터 처리에, `scikit-learn`은 전처리·학습·평가에 사용합니다.

In [ ]:
from pathlib import Path
from zipfile import ZipFile

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score

## 2. 데이터 불러오기

저장소 루트 또는 `baseline` 폴더에서 실행합니다. ZIP 안의 CSV를 바로 읽으므로 압축을 풀 필요가 없습니다.

In [ ]:
project_dir = Path.cwd()
if not (project_dir / "data/dataset.zip").exists():
    project_dir = project_dir.parent

with ZipFile(project_dir / "data/dataset.zip") as dataset:
    train = pd.read_csv(dataset.open("train.csv"))
    test = pd.read_csv(dataset.open("test.csv"))
    sample_submission = pd.read_csv(dataset.open("sample_submission.csv"))

print("train:", train.shape, "test:", test.shape)
train.head()

## 3. 입력 X와 정답 y 분리

`target`은 정답이고, `id`는 제출 행을 구분하는 식별자입니다. 두 열을 모델 입력에서 제외합니다.

In [ ]:
X = train.drop(columns=["id", "target"])
y = train["target"]
X_test = test[X.columns]

## 4. 학습/검증 데이터 분리

80%는 학습, 20%는 검증에 사용합니다. `stratify=y`는 정답 비율을 비슷하게 유지하고, `random_state`는 결과 재현을 돕습니다. 단순 무작위 분할이므로 유사한 행이 양쪽에 포함되어 검증 점수가 낙관적일 수 있습니다.

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

## 5. 기본 전처리와 모델 구성

숫자형 결측값은 중앙값으로 채우고 크기를 표준화합니다. 범주형 결측값은 최빈값으로 채우고 원-핫 인코딩으로 숫자로 바꿉니다. Pipeline을 사용하면 전처리 통계도 학습 데이터에서만 계산됩니다. 모델은 기본 로지스틱 회귀 하나를 사용하며 튜닝하지 않습니다.

In [ ]:
numeric_columns = X_train.select_dtypes(include="number").columns
categorical_columns = X_train.select_dtypes(exclude="number").columns

numeric_preprocessing = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_preprocessing = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])
preprocessing = ColumnTransformer([
    ("numeric", numeric_preprocessing, numeric_columns),
    ("categorical", categorical_preprocessing, categorical_columns),
])
model = Pipeline([
    ("preprocessing", preprocessing),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])

## 6. 학습 및 validation 점수

Balanced Accuracy는 0과 1 각각의 재현율을 평균한 값입니다. 스터디 점수는 이 값에 100을 곱합니다. `predict`는 기본 기준으로 0 또는 1을 반환합니다.

In [ ]:
model.fit(X_train, y_train)
valid_prediction = model.predict(X_valid)
score = 100 * balanced_accuracy_score(y_valid, valid_prediction)
print(f"Validation 점수: {score:.2f} / 100")

## 7. test 예측

검증을 마친 동일한 모델을 전체 train 데이터로 다시 학습한 뒤 test를 예측합니다.

In [ ]:
model.fit(X, y)
test_prediction = model.predict(X_test)

## 8. submission.csv 생성

예시 제출 파일의 ID 순서를 유지합니다. `target`에는 확률이 아닌 정수 0 또는 1을 저장합니다. 결과 파일은 저장소 루트에 만들어집니다.

In [ ]:
prediction_by_id = pd.Series(test_prediction, index=test["id"])
submission = sample_submission.copy()
submission["target"] = submission["id"].map(prediction_by_id).astype(int)

assert list(submission.columns) == ["id", "target"]
assert submission["id"].is_unique
assert set(submission["id"]) == set(test["id"])
assert submission["target"].isin([0, 1]).all()

submission.to_csv(project_dir / "submission.csv", index=False)
print("submission.csv 저장 완료:", submission.shape)
submission.head()